<a href="https://colab.research.google.com/github/12yuuuu/12yuuuu/blob/main/MNIST_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np  # For numerical operations
import pandas as pd  # For data processing and CSV file I/O
import struct  # For parsing and packaging of binary data

from sklearn.datasets import make_classification  # For generating synthetic datasets for classification
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis  # For Quadratic Discriminant Analysis classifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier  # For ensemble methods like AdaBoost and Random Forest
from sklearn.gaussian_process import GaussianProcessClassifier  # For Gaussian Process classifier
from sklearn.gaussian_process.kernels import RBF  # Radial Basis Function kernel for Gaussian Process classifier
from sklearn.naive_bayes import GaussianNB  # For Gaussian Naive Bayes classifier
from sklearn.neighbors import KNeighborsClassifier  # For K-Nearest Neighbors classifier
from sklearn.neural_network import MLPClassifier  # For Multi-layer Perceptron classifier
from sklearn.pipeline import make_pipeline  # For constructing pipelines
from sklearn.preprocessing import StandardScaler  # For standardizing features
from sklearn.svm import SVC  # For Support Vector Machine classifier
from sklearn.tree import DecisionTreeClassifier  # For Decision Tree classifier
from sklearn.linear_model import LogisticRegression  # For Logistic Regression classifier
from xgboost import XGBClassifier  # For XGBoost classifier
from sklearn.model_selection import train_test_split  # For splitting datasets into train and test sets
from sklearn.metrics import accuracy_score  # For evaluating the accuracy of classifiers

import tensorflow as tf  # Import TensorFlow library for deep learning tasks
from tensorflow.keras import layers, models  # Import specific modules for building neural network models
from tensorflow.keras.datasets import mnist  # Import MNIST dataset for training and testing
from tensorflow.keras.utils import to_categorical  # Import utility function for one-hot encoding of labels

from tabulate import tabulate  # Import the tabulate function from the tabulate library for creating tables

In [2]:
import os

# This script walks through the directory '/content/input' and prints the full path of each file found
for dirname, _, filenames in os.walk('/content'):
  for filename in filenames:
    print(os.path.join(dirname, filename))

/content/t10k-labels.idx1-ubyte
/content/t10k-images.idx3-ubyte
/content/train-images.idx3-ubyte
/content/train-labels.idx1-ubyte
/content/.config/active_config
/content/.config/gce
/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/.last_survey_prompt.yaml
/content/.config/default_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/logs/2024.06.03/13.34.53.444249.log
/content/.config/logs/2024.06.03/13.22.06.669614.log
/content/.config/logs/2024.06.03/13.34.41.044819.log
/content/.config/logs/2024.06.03/13.34.52.780168.log
/content/.config/logs/2024.06.03/13.29.11.352820.log
/content/.config/logs/2024.06.03/13.28.59.852590.log
/content/.config/configurations/config_default


In [3]:
import warnings

# This script configures the warnings module to ignore specific warnings to keep the output clean and focused.
warnings.filterwarnings("ignore") # Ignore all warnings
warnings.filterwarnings("ignore", message=".*divide by zero.*") # Ignore specific warnings by message
warnings.filterwarnings("ignore", category=DeprecationWarning) # Ignore specific warnings by category

In [4]:
def read_idx(filename):
  """
  This function reads an IDX file and converts it into a NumPy array for easy manipulation and analysis.

  Parameters:
  filename (str): The path to the IDX file to be read.

  Returns:
  np.ndarray: A NumPy array containing the data from the IDX file.
  """
  with open(filename, 'rb') as f:
    # Read the first 4 bytes to get the magic number
    # '>HBB' is a format string, indicating that big-endian (>) is used to read a 16-bit (2-byte) integer (H), and two 8-bit (1-byte) integer (B)
    # You will get three values ​​here: zero (should be 0), data_type (data type), dims (number of dimensions)
    magic_number = f.read(4)
    zero, data_type, dims = struct.unpack('>HBB', magic_number)

    # Read the next 'dims' * 4 bytes to get the shape of the data
    # '>I' is a format string, indicating reading a 32-bit (4-byte) integer in big-endian order
    shape = tuple(struct.unpack('>I', f.read(4))[0] for d in range(dims))

    # Read the rest of the file and convert it to a numpy array with the specified shape
    return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

In [14]:
def load_mnist(image_path, label_path):
  """
  This function loads the MNIST dataset from the specified image and label IDX file paths.

  Parameters:
  image_path (str): The path to the IDX file containing the images.
  label_path (str): The path to the IDX file containing the labels.

  Returns:
  tuple: A tuple containing two NumPy arrays: the first is the images array, and the second is the labels array.
  """
  images = read_idx(image_path)
  labels = read_idx(label_path)
  return images, labels

# Paths to the MNIST dataset files
train_image_path = '/content/train-images.idx3-ubyte'
train_label_path = '/content/train-labels.idx1-ubyte'
test_image_path = '/content/t10k-images.idx3-ubyte'
test_label_path = '/content/t10k-labels.idx1-ubyte'

In [15]:
# Load the training and testing data
train_images, train_labels = load_mnist(train_image_path, train_label_path)
test_images, test_labels = load_mnist(test_image_path, test_label_path)

# Display the shapes of the loaded data
print(f"Training images shape: {train_images.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Testing images shape: {test_images.shape}")
print(f"Testing labels shape: {test_labels.shape}")

Training images shape: (60000, 28, 28)
Training labels shape: (60000,)
Testing images shape: (10000, 28, 28)
Testing labels shape: (10000,)


In [16]:
# This script reshape the training and testing images into flat arrays, and keep the labels as 1D arrays.
train_images_flat = train_images.reshape(train_images.shape[0], -1) # Reshape the training images to 2D arrays with shape (60000, 784)
test_images_flat = test_images.reshape(test_images.shape[0], -1) # Reshape the testing images to 2D arrays with shape (10000, 784)

# Assign the images to X_train, y_train, X_test, y_test
X_train = train_images_flat
y_train = train_labels
X_test = test_images_flat
y_test = test_labels

In [17]:
# Define different classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'SupportVectorMachine': make_pipeline(StandardScaler(), SVC()),
    'MultiLayerPerceptron': make_pipeline(StandardScaler(), MLPClassifier(max_iter=1000)),
    'LogisticRegression': LogisticRegression(),
    'XgboostClassifier': XGBClassifier()
}

# Train and evaluate each classifier
accuracies = {}

for name, clf in classifiers.items():
    # train classifier
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculation accuracy
    accuracy = accuracy_score(y_test, y_pred)
    # Storage accuracy
    accuracies[name] = accuracy
    print(f"{name}: {accuracy:.4f}")

# Output the accuracy of all classifiers
print("\nAccuracies of different classifiers:")
for name, accuracy in accuracies.items():
    print(f"{name}: {accuracy:.4f}")

RandomForestClassifier: 0.9708
KNeighborsClassifier: 0.9688
SupportVectorMachine: 0.9660
MultiLayerPerceptron: 0.9748
LogisticRegression: 0.9255
XgboostClassifier: 0.9795

Accuracies of different classifiers:
RandomForestClassifier: 0.9708
KNeighborsClassifier: 0.9688
SupportVectorMachine: 0.9660
MultiLayerPerceptron: 0.9748
LogisticRegression: 0.9255
XgboostClassifier: 0.9795


In [18]:
# Load and preprocess the MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize the input images to the range [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape the data to include the channel dimension (required for CNNs)
# Assuming images are 28x28 pixels
X_train = X_train.reshape((X_train.shape[0], 28, 28, 1))
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1))
y_train = to_categorical(y_train, 10)  # Convert labels to one-hot encoding
y_test = to_categorical(y_test, 10)  # Convert labels to one-hot encoding

# Verify the shape of the data
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

# Define the CNN model
model = models.Sequential()

# First convolutional layer
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))

# Second convolutional layer
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# Third convolutional layer
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Flatten the output from the convolutional layers
model.add(layers.Flatten())

# Fully connected layer
model.add(layers.Dense(64, activation='relu'))

# Output layer with softmax activation for classification
model.add(layers.Dense(10, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.2)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')

11490434/11490434 [==============================] - 0s 0us/step
Shape of X_train: (60000, 28, 28, 1)
Shape of y_train: (60000, 10)
Shape of X_test: (10000, 28, 28, 1)
Shape of y_test: (10000, 10)
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 64)          0         
 g2D)                                                            
                                                         

In [19]:
# Load and preprocess the MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize the input images to the range [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape the data to include the channel dimension (required for CNNs)
# Assuming images are 28x28 pixels
X_train = X_train.reshape((X_train.shape[0], 28, 28, 1))
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1))
y_train = to_categorical(y_train, 10)  # Convert labels to one-hot encoding
y_test = to_categorical(y_test, 10)  # Convert labels to one-hot encoding

# Verify the shape of the data
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

# Define the CNN model
model = models.Sequential()

# First convolutional layer
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))

# Second convolutional layer
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# Third convolutional layer
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Flatten the output from the convolutional layers
model.add(layers.Flatten())

# Fully connected layer
model.add(layers.Dense(64, activation='relu'))

# Output layer with softmax activation for classification
model.add(layers.Dense(10, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.2)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')

Shape of X_train: (60000, 28, 28, 1)
Shape of y_train: (60000, 10)
Shape of X_test: (10000, 28, 28, 1)
Shape of y_test: (10000, 10)
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 13, 13, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 5, 5, 64)          0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 3, 3, 64)         

In [20]:
# This function creates a formatted table from a dictionary of model names and their accuracies, including the CNN model's accuracy.
# The accuracies are multiplied by 100 for better readability.
def create_table(accuracies):
    """
    Creates a table from a dictionary of model names and their accuracies.

    Args:
      accuracies: A dictionary with model names as keys and accuracies as values.

    Returns:
      A string containing the formatted table.
    """

    # Multiply all values by 100
    accuracies = {name: accuracy * 100 for name, accuracy in accuracies.items()}

    # Add CNN and test_acc
    accuracies["CNN"] = test_acc * 100

    # Create the table
    table = tabulate(accuracies.items(), headers=["Model", "Accuracy (%)"], tablefmt="grid")

    return table

# Create and display the table
table = create_table(accuracies)
print(table)

+------------------------+----------------+
| Model                  |   Accuracy (%) |
+========================+================+
| RandomForestClassifier |          97.08 |
+------------------------+----------------+
| KNeighborsClassifier   |          96.88 |
+------------------------+----------------+
| SupportVectorMachine   |          96.6  |
+------------------------+----------------+
| MultiLayerPerceptron   |          97.48 |
+------------------------+----------------+
| LogisticRegression     |          92.55 |
+------------------------+----------------+
| XgboostClassifier      |          97.95 |
+------------------------+----------------+
| CNN                    |          99.29 |
+------------------------+----------------+
